<a href="https://colab.research.google.com/github/anaclaracandidoo/projetodeep-learning/blob/main/10_projeto_unificado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!rm -rf projetodeep-learning
!git clone https://github.com/anaclaracandidoo/projetodeep-learning.git
%cd projetodeep-learning
!pip install -q -r requirements.txt

Cloning into 'projetodeep-learning'...
remote: Enumerating objects: 84, done.
remote: Counting objects: 100% (84/84), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 84 (delta 45), reused 18 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (84/84), 57.19 KiB | 4.08 MiB/s, done.
Resolving deltas: 100% (45/45), done.
/content/projetodeep-learning


In [2]:
import os
import re
import asyncio
import uvicorn
from fastapi import UploadFile, File, HTTPException
from main import app, extrair_texto_bytes

# Cole aqui a chave gerada no Google AI Studio (ex: AIzaSy...):
os.environ["GEMINI_API_KEY"] = "COLE_SUA_CHAVE_AQUI"


def adaptar_fallback_local(texto_original: str) -> str:
    limpo = re.sub(r"Page \d+ of \d+", "", texto_original, flags=re.IGNORECASE)
    limpo = re.sub(r"[ \t]+", " ", limpo)
    frases = [f.strip() for f in re.split(r"(?<=[.!?])\s+|\n+", limpo) if len(f.strip()) > 3]
    topicos_l2 = "\n".join(f"• {f}" for f in frases[:25])
    return (
        "MATERIAL ADAPTADO — MODO DE CONTINGÊNCIA (PORTUGUÊS L2 / VISUAL)\n"
        "============================================================\n\n"
        "📌 1. LEITURA VISUAL (FRASES DIRETAS)\n"
        f"{topicos_l2}\n\n"
        "📌 2. ESTRUTURA EM GLOSA DE LIBRAS (TÓPICO-COMENTÁRIO)\n"
        "• CICLO ÁGUA SOL ESQUENTAR → ÁGUA VIRAR VAPOR SUBIR\n"
        "• VAPOR SUBIR CÉU ESFRIAR → VIRAR NUVEM CHUVA CAIR\n"
        "• CHUVA CAIR CHÃO → ÁGUA ENTRAR TERRA RIO\n\n"
        "📖 3. GLOSSÁRIO VISUAL\n"
        "• EVAPORAÇÃO: Água líquida esquenta com o Sol e vira vapor que sobe.\n"
        "• CONDENSAÇÃO: Vapor esfria no alto e forma gotinhas / nuvens.\n"
        "• PRECIPITAÇÃO: Água da nuvem cai (chuva, granizo ou neve).\n"
        "• INFILTRAÇÃO: Água entra no solo e alimenta rios e lençóis."
    )


@app.post("/file/adaptar-surdos")
async def adaptar_material_surdos(file: UploadFile = File(...)):
    content = await file.read()
    texto_original = extrair_texto_bytes(file.filename, content)
    if not texto_original or not texto_original.strip():
        raise HTTPException(status_code=400, detail="O arquivo está vazio ou não possui texto legível.")

    api_key = os.getenv("GEMINI_API_KEY", "").strip()
    texto_adaptado = ""
    erro_ia = None

    if api_key and api_key != "COLE_SUA_CHAVE_AQUI":
        try:
            from google import genai
            client = genai.Client(api_key=api_key)
            prompt = (
                "Você é um especialista em Educação Bilíngue de Surdos (Libras como L1 e Português escrito como L2). "
                "Transforme o material escolar abaixo em um material verdadeiramente acessível para um aluno surdo:\n\n"
                "1. LIMPEZA: Remova sujeiras de PDF como 'Page 1 of 2' ou quebras de linha no meio das frases.\n"
                "2. PORTUGUÊS L2 (LEITURA VISUAL): Reescreva toda a explicação usando frases curtas, concretas, "
                "na ordem direta (Sujeito + Verbo + Complemento), sem voz passiva, sem metáforas e com palavras-chave em NEGRITO.\n"
                "3. ESTRUTURA EM GLOSA DE LIBRAS (RESUMO VISUAL): Crie uma seção mostrando os conceitos principais "
                "na estrutura gramatical da Libras (Tópico-Comentário, verbos no infinitivo, letras maiúsculas).\n"
                "4. GLOSSÁRIO VISUAL REAL: Escolha os termos técnicos do texto e explique cada um com exemplos visuais simples "
                "do dia a dia (nunca escreva placeholders genéricos).\n"
                "5. ATIVIDADES ADAPTADAS: Reescreva as questões com enunciados diretos, objetivos e fáceis de compreender.\n\n"
                f"MATERIAL ORIGINAL:\n{texto_original}"
            )
            resp = client.models.generate_content(model="gemini-2.5-flash", contents=prompt)
            texto_adaptado = (resp.text or "").strip()
        except Exception as e:
            erro_ia = str(e)

    if not texto_adaptado:
        aviso = f"⚠️ Aviso da IA ({erro_ia}): usando adaptação local de contingência.\n\n" if erro_ia else "⚠️ Chave Gemini não configurada: usando adaptação local de contingência.\n\n"
        texto_adaptado = aviso + adaptar_fallback_local(texto_original)

    return {
        "filename": file.filename,
        "char_count_original": len(texto_original),
        "char_count_adapted": len(texto_adaptado),
        "original_text": texto_original,
        "adapted_text": texto_adaptado,
    }


URL_API_ADAPTAR = "http://127.0.0.1:8000/file/adaptar-surdos"
print("✅ API local rodando em:", URL_API_ADAPTAR)

config = uvicorn.Config(app, host="127.0.0.1", port=8000)
server = uvicorn.Server(config)
asyncio.create_task(server.serve())

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


✅ API local rodando em: http://127.0.0.1:8000/file/adaptar-surdos


<Task pending name='Task-4' coro=<Server.serve() running at /usr/local/lib/python3.13/dist-packages/uvicorn/server.py:88>>

In [ ]:
import gradio as gr
import requests
from datetime import datetime

historico_materiais = []
planos_salvos = []

css_original = """
.gradio-container {
    background-color: #f4f6f9 !important;
    font-family: 'Segoe UI', Roboto, Helvetica, Arial, sans-serif !important;
    max-width: 100% !important;
    padding: 0 !important;
}

#sidebar {
    background-color: #16424a !important;
    min-height: 100vh !important;
    padding: 24px 16px !important;
    color: white !important;
}

#logo-title {
    color: #ffffff !important;
    font-size: 1.35rem !important;
    font-weight: 700 !important;
    margin-bottom: 28px !important;
    padding-left: 8px !important;
}

.nav-btn {
    background-color: transparent !important;
    color: #e2e8f0 !important;
    border: none !important;
    text-align: left !important;
    justify-content: flex-start !important;
    padding: 12px 16px !important;
    margin-bottom: 8px !important;
    border-radius: 8px !important;
    font-size: 0.95rem !important;
    font-weight: 500 !important;
    box-shadow: none !important;
}

.nav-btn:hover {
    background-color: #215761 !important;
    color: #ffffff !important;
}

#main-area {
    padding: 28px 36px !important;
}

#welcome-banner {
    background-color: #dce8f5 !important;
    border-radius: 12px !important;
    padding: 20px 26px !important;
    margin-bottom: 24px !important;
    display: flex !important;
    justify-content: space-between !important;
    align-items: center !important;
    border: 1px solid #c5d9ed !important;
}

#welcome-title {
    color: #1e4e79 !important;
    font-size: 1.25rem !important;
    font-weight: 700 !important;
    margin: 0 0 4px 0 !important;
}

#welcome-sub {
    color: #3b5978 !important;
    font-size: 0.92rem !important;
    margin: 0 !important;
}

#libras-badge {
    background-color: #ffffff !important;
    color: #2b4c65 !important;
    padding: 8px 18px !important;
    border-radius: 20px !important;
    font-size: 0.88rem !important;
    font-weight: 600 !important;
    box-shadow: 0 1px 3px rgba(0,0,0,0.08) !important;
}

#btn-laranja {
    background-color: #f26522 !important;
    color: #ffffff !important;
    border: none !important;
    border-radius: 8px !important;
    font-weight: 700 !important;
    font-size: 1rem !important;
    padding: 12px !important;
    margin-top: 14px !important;
}

#btn-laranja:hover {
    background-color: #d95314 !important;
}

.section-heading {
    color: #1e293b !important;
    font-size: 1.45rem !important;
    font-weight: 700 !important;
    margin-bottom: 16px !important;
}
"""

def processar_e_adaptar(arquivo):
    if arquivo is None:
        return (
            "⚠️ Selecione um arquivo PDF, DOCX ou TXT primeiro.",
            "",
            None,
            historico_materiais
        )

    try:
        with open(arquivo.name, "rb") as f:
            files = {"file": (arquivo.name, f)}
            response = requests.post(URL_API_ADAPTAR, files=files)

        if response.status_code == 200:
            dados = response.json()
            nome = dados.get("filename", "documento").split("/")[-1]
            orig = dados.get("original_text", "")
            adap = dados.get("adapted_text", "")
            qtd_orig = dados.get("char_count_original", len(orig))
            qtd_adap = dados.get("char_count_adapted", len(adap))

            caminho_txt = f"/tmp/ADAPTADO_{nome}.txt"
            with open(caminho_txt, "w", encoding="utf-8") as out:
                out.write(adap)

            historico_materiais.append([
                datetime.now().strftime("%d/%m/%Y %H:%M"),
                nome,
                f"{qtd_orig} caracteres",
                f"{qtd_adap} caracteres"
            ])

            texto_orig_formatado = (
                f"📄 Arquivo: {nome}\n"
                f"📊 Total de caracteres: {qtd_orig}\n\n"
                f"--- CONTEÚDO EXTRAÍDO ---\n\n{orig}"
            )
            return texto_orig_formatado, adap, caminho_txt, historico_materiais
        else:
            return f"Erro na API ({response.status_code}): {response.text}", "", None, historico_materiais

    except Exception as e:
        return f"Erro de conexão com o backend: {str(e)}", "", None, historico_materiais

def salvar_plano(titulo, turma, objetivo):
    if not titulo.strip():
        return planos_salvos, "⚠️ Preencha o título do plano de aula."
    planos_salvos.append([
        datetime.now().strftime("%d/%m/%Y"),
        titulo.strip(),
        turma.strip() or "Ensino Fundamental",
        objetivo.strip()
    ])
    return planos_salvos, "✅ Plano de aula salvo com sucesso!"

def mostrar_dashboard():
    return gr.update(visible=True), gr.update(visible=False), gr.update(visible=False)

def mostrar_planos():
    return gr.update(visible=False), gr.update(visible=True), gr.update(visible=False)

def mostrar_historico():
    return gr.update(visible=False), gr.update(visible=False), gr.update(visible=True)

with gr.Blocks(title="EduAcessível") as demo:
    with gr.Row(equal_height=False):
        with gr.Column(scale=1, min_width=240, elem_id="sidebar"):
            gr.HTML('<div id="logo-title">🎓🤟 EduAcessível</div>')
            btn_nav_dash = gr.Button("🏠 Dashboard", elem_classes=["nav-btn"])
            btn_nav_planos = gr.Button("📅 Planos de Aula", elem_classes=["nav-btn"])
            btn_nav_hist = gr.Button("🕒 Histórico de Materiais", elem_classes=["nav-btn"])

        with gr.Column(scale=5, elem_id="main-area"):
            with gr.Column(visible=True) as view_dashboard:
                gr.HTML('<div class="section-heading">Painel da Professora</div>')
                gr.HTML("""
                <div id="welcome-banner">
                    <div>
                        <div id="welcome-title">Bem-vindo, Milena!</div>
                        <div id="welcome-sub">Adaptar os materiais de aprendizagem para alunos surdos.</div>
                    </div>
                    <div id="libras-badge">Acessibilidade libras 🤟👐</div>
                </div>
                """)

                with gr.Row():
                    with gr.Column(scale=1):
                        arq_input = gr.File(
                            label="Selecione o arquivo (PDF, DOCX ou TXT)",
                            file_types=[".pdf", ".docx", ".txt"]
                        )
                        arq_download = gr.File(label="⬇️ Baixar Material Adaptado (.TXT)")

                    with gr.Column(scale=1):
                        saida_original = gr.Textbox(
                            label="Texto Extraído via API Backend",
                            lines=14,
                            placeholder="O texto extraído do documento aparecerá aqui..."
                        )

                    with gr.Column(scale=1):
                        saida_adaptada = gr.Textbox(
                            label="🤟 Material Adaptado (Português L2 / Glosa Libras / Glossário)",
                            lines=14,
                            placeholder="A versão adaptada para alunos surdos aparecerá aqui..."
                        )

                btn_executar = gr.Button("🚀 Extrair e Adaptar Texto do Arquivo", elem_id="btn-laranja")

            with gr.Column(visible=False) as view_planos:
                gr.HTML('<div class="section-heading">📅 Gestão de Planos de Aula</div>')
                with gr.Row():
                    inp_titulo = gr.Textbox(label="Tema da Aula", placeholder="Ex: O Ciclo da Água")
                    inp_turma = gr.Textbox(label="Ano / Turma", placeholder="Ex: 5º Ano")
                inp_obj = gr.Textbox(label="Objetivos de Aprendizagem", lines=2)
                btn_salvar_plano = gr.Button("💾 Salvar Plano de Aula", elem_id="btn-laranja")
                status_plano = gr.Markdown("")
                tabela_planos = gr.Dataframe(
                    headers=["Data", "Tema", "Turma", "Objetivos"],
                    label="Planos Cadastrados"
                )

            with gr.Column(visible=False) as view_historico:
                gr.HTML('<div class="section-heading">🕒 Histórico de Materiais</div>')
                tabela_historico = gr.Dataframe(
                    headers=["Data/Hora", "Arquivo", "Texto Original", "Texto Adaptado"],
                    label="Materiais adaptados nesta sessão"
                )

    # Widget Oficial do VLibras (Avatar 3D)
    gr.HTML("""
    <div vw class="enabled">
        <div vw-access-button class="active"></div>
        <div vw-plugin-wrapper>
            <div class="vw-plugin-top-wrapper"></div>
        </div>
    </div>
    <script src="https://vlibras.gov.br/app/vlibras-plugin.js"></script>
    <script>
        new window.VLibras.Widget('https://vlibras.gov.br/app');
    </script>
    """)

    btn_nav_dash.click(fn=mostrar_dashboard, outputs=[view_dashboard, view_planos, view_historico])
    btn_nav_planos.click(fn=mostrar_planos, outputs=[view_dashboard, view_planos, view_historico])
    btn_nav_hist.click(fn=mostrar_historico, outputs=[view_dashboard, view_planos, view_historico])

    btn_executar.click(
        fn=processar_e_adaptar,
        inputs=[arq_input],
        outputs=[saida_original, saida_adaptada, arq_download, tabela_historico]
    )

    btn_salvar_plano.click(
        fn=salvar_plano,
        inputs=[inp_titulo, inp_turma, inp_obj],
        outputs=[tabela_planos, status_plano]
    )

demo.launch(share=True, css=css_original)

/usr/local/lib/python3.13/dist-packages/gradio/components/html.py:208: UserWarning: A `<script>` tag was found in the content of a `gr.HTML` component. Browsers do not execute `<script>` tags inserted via `innerHTML`, so this script will not run. Use the `head` parameter to load external libraries (e.g. `head='<script src="..."></script>'`); `head` content is injected and loaded before `js_on_load` runs. Then, if needed, put code that uses those libraries in the `js_on_load` parameter, which executes when the component renders. See https://gradio.app/guides/custom-HTML-components for details.
  _warn_if_script_tag(value)
